The goal of this notebook is to create a dataset of the site-specific variables that I want to use as parameters in my regression analyses in 073. For more info on the regression analyses see admin/analysis_models.md

Parameters:
- hazard curve parameters
    - k0, k1, k2 from the parabolic hazard curve fit: "C:\Users\clemettn\Documents\phd\data_processed\03_site_hazard\AvgSA_03_hazard_fit_second_order_60sites_4sig.csv"
- Period: T1 -> this should be the period obtained from modal analysis and not from the design data. Notify me if you don't have it. 
- spectral shape: SA(T1) / AvgSA[0,3] -> at RTP of the AvgSA imls closest to a probability of collapse of 0.2, 0.5, and 0.8 as well as at the design level RTP 475 years. SA(T1) can be obtained from the hazard curves "C:\Users\clemettn\Documents\phd\data_processed\03_site_hazard\SA_hazard_curves_60sites_4sig.pickle"
- spectral shape: SA(T1) / AvgSA[0:0.01:3]   -> different definition of AvgSA (0.01s spacing between ordinates) -> interpolate values from get from the SA hazard curves using logspace interpolation. 4 versions same as above
- Conditional Mean RSD595 for the site -> 4 versions same as above
- Conditional standard deviation of the RSD595 at the site -> 4 versions same as above

For the 4 variations consider all the imls that were disaggregated not just those that were analyses. The mean and standard deviation of RSD595 can be obtained from the GCIM distribtuions for many of the imls. the distributions already computed are here: C:\Users\clemettn\Documents\phd\data_processed\05_gcim_distributions. If some are needed but not yet computed then compute them. Resave the exisitng ones that you use and any newly computed ones in "C:\Users\clemettn\Documents\phd\data_processed\10_fragility_curve_statistical_analysis\regression_coefficients". Use this folder for any new intermediate data that you produce. 

The final parameter data set should be a dataframe with each row corresponding to one of the building/site combinations. The columns are for the different parameters. It should be saved here C:\Users\clemettn\Documents\phd\data_processed\10_fragility_curve_statistical_analysis

Don't create any new modules to start with. Put all teh working in this notebook

# 078 — Site regression variables

This notebook builds the **120-row covariate table** used by the meta-regressions in
`073-statistical_models.ipynb` (A3, and later A0b / A11; see `admin/analysis_models.md` §A3).
There is one row per (site, structure), so 60 sites × {3-storey, 5-storey}, and one column per covariate.
Downstream notebooks join it on `["site", "n_storeys"]`.

## What each covariate measures

| group | columns | scope | why it might matter |
|---|---|---|---|
| structure period | `T1` (modal, from the OpenSees model), `T1_design` (design-script elastic model) | per design group | sets where on the spectrum the structure "listens" |
| hazard-curve shape | `k0, k1, k2` (second-order fit to the AvgSA$_{0-3}$ hazard curve) | per site | slope and curvature of the hazard; SAC/FEMA-style risk integrals depend on them |
| hazard level | `avgsa03_rtp{475,2500,5000,10000}` | per site | how strong the shaking is at fixed return periods |
| spectral shape (UHS) | `sa_t1_avgsa03_L`, `saratio_uhs_L` | per row and level | how "peaked" the hazard spectrum is at T1 |
| spectral shape (GCIM) | `ln_sa_t1_avgsa03_cm_L`, `ln_saratio_cm_L` | per row and level | the same idea, but from the spectrum you *expect* given AvgSA = level |
| duration | `ln_rsd595_mean_L`, `ln_rsd595_sigma_L` | per site and level | long records damage more at the same spectral intensity |

**Levels `L`.** Each level-dependent quantity is evaluated at four AvgSA$_{0-3}$ intensity levels:

* `pc20`, `pc50`, `pc80`: the **disaggregated** IML (any of the ~26 per site, not only the analysed
  stripes) closest in log space to the row's MSA site-specific fragility quantile
  $\mathrm{IM}_p = \theta\,e^{\beta\,\Phi^{-1}(p)}$ for $p = 0.2, 0.5, 0.8$;
* `rtp475`: the design-level intensity, i.e. AvgSA$_{0-3}$ at a 475-yr return period.

## The two spectral-shape definitions

**Ratio to the conditioning IM:** $\;\mathrm{SA}(T_1)/\mathrm{AvgSA}_{0-3}$, where
$\mathrm{AvgSA}_{0-3}$ is the level itself.

**SaRatio.** This is Zhong et al. (2022), *Earthquake Spectra* 38(3), Eq. 2, p. 1897 (page approximate). They take it
from Eads et al. (2016):

$$\mathrm{SaRatio}(T_a, T_1, T_b) = \frac{\mathrm{SA}(T_1)}{\left[\prod_{i=1}^{n}\mathrm{SA}(T_i)\right]^{1/n}},
\qquad T_i \in [T_a : 0.01\,\mathrm{s} : T_b],\quad T_a = 0.2T_1,\; T_b = 3T_1 .$$

Think of it as "how tall is the spectrum at $T_1$ compared with its average height over the band the
structure moves through as it softens". A **low** SaRatio means the spectrum falls away slowly beyond $T_1$, which
is the more damaging case (Zhong et al. 2022, Fig. 2a).

Each shape quantity is computed **two ways**:

1. **UHS-based** (`*_uhs`, `sa_t1_*`): every ordinate is read from its own SA hazard curve at the level's return
   period. This follows the brief. Caveat: a UHS envelopes different earthquakes at different periods, so the
   ratio of two UHS ordinates is not the shape of any one ground motion (Baker 2011, *J. Struct. Eng.* 137:322).
   SaRatio needs SA up to $3T_1 \approx 2.7$ s, but the existing SA hazard stops at 1.2 s. **The SA PSHA is
   therefore re-run to 3 s** (section 5).
2. **Conditional-mean** (`*_cm`): this is what Zhong et al. actually do for their site-specific targets (§ "Case
   study", Table 2, ~p. 1901–1904). They take the GCIM conditional means $\mu_{\ln SA(T)\mid \mathrm{IM}}$
   (Bradley 2010) and form
   $E[\ln \mathrm{SaRatio}] = \mu_{\ln SA(T_1)} - \tfrac1n\sum_i \mu_{\ln SA(T_i)}$.
   This is exact under the GCIM mixture because expectation is linear. It needs no new PSHA. The GCIM already
   carries 20 SA periods from 0.025 to 3 s, conditioned on AvgSA$_{0-3}$.

## Duration

`ln_rsd595_mean_L` and `ln_rsd595_sigma_L` are the GCIM mixture mean and standard deviation of
$\ln D_{s5-95}$ given AvgSA$_{0-3}$ = level (Bahrampouri et al. 2021 GMMs). The 475-yr level is not on the
disaggregation grid, so it needs **a new disaggregation** (section 5).

## OpenQuake work — run on the OQ machine, not here

Section 5 **always writes** the two job files and **runs them only when `RUN_OQ = True`**. Each job goes through
`oq_runner.run_or_reuse`, so it is launched only when it has no manifest entry, when its inputs (config, logic
trees, site model, source models) have changed, or when its datastore has gone. The same cell extracts the results
to two small pickles in `regression_coefficients/`, each stamped with its input fingerprint and `calc_id`. Copy
those two pickles to this machine. Every section after 5 reads **only the pickles**. When a pickle is absent or
stale, the dependent columns come out NaN, and the notebook says which file to bring over.

## Prerequisite

`dvc pull` the GCIM pickle, the disaggregation shards and the disaggregation stats. The local copies can predate
the 0.3434 g disaggregation. Section 6 refuses to pick IML levels from an incomplete grid
(`REQUIRE_CURRENT_DISAGG`).

In [ ]:
%load_ext autoreload
%autoreload 2

## 0. Setup & parameters

In [ ]:
import json
import pickle
import re
from pathlib import Path
from string import Template

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

from phd_project.config import config
from phd_project.scripts import oq_runner, oqhelpers
from phd_project.scripts import fragility_data_models as fm
from phd_project.scripts import metaregression_analysis as mra
from phd_project.scripts.cache_utils import (
    fingerprint, load_or_compute, manifest_matches, write_manifest)
from phd_project.scripts.disagg_shards import load_shards, read_index
from phd_project.scripts.msa_ida_hypothesis_testing import site_hazard_at_return_periods

cfg = config.load_config()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

In [ ]:
# -----------------------------------------------------------------------------
# PARAMETERS
# -----------------------------------------------------------------------------
IM_TAG = "AvgSA_03"     # conditioning IM of the whole study (AvgSA over 0-3 s)
IMT = "AvgSA"           # its key in the hazard-curve / disaggregation dicts

# Study dimensions (CLAUDE.md s.1): 60 sites x {3s, 5s} = 120 rows, 51 design groups.
N_STOREYS = [3, 5]
N_SITES, N_ROWS, N_GROUPS = 60, 120, 51

# Intensity levels at which every level-dependent covariate is evaluated.
# PC_LEVELS: probability of collapse on the row's MSA site-specific fragility.
PC_LEVELS = {"pc20": 0.2, "pc50": 0.5, "pc80": 0.8}
DESIGN_RTP = 475                                   # design-level return period [yr]
DESIGN_LEVEL = f"rtp{DESIGN_RTP}"
LEVELS = [*PC_LEVELS, DESIGN_LEVEL]

# AvgSA_03 hazard levels reported as site covariates [yr].
HAZARD_RTPS = (475, 2500, 5000, 10000)

# SaRatio band, Zhong et al. (2022) Eq. 2 after Eads et al. (2016): Ta = 0.2 T1,
# Tb = 3 T1, sampled every 0.01 s.
SARATIO_TA, SARATIO_TB, SARATIO_DT = 0.2, 3.0, 0.01

# Periods of the re-run SA PSHA: PGA + 0.05 ... 3.00 s every 0.05 s. The old run
# stopped at 1.2 s, short of 3 T1 (up to ~2.7 s) for the SaRatio band.
SA_PSHA_PERIODS = np.round(np.arange(0.05, 3.0 + 1e-9, 0.05), 2)
TRUNCATION = 4          # epsilon truncation of every hazard product used here

# MSA-SS fragilities not refitted after an extra stripe ran (memory note,
# 2026-09-15). Flagged, not dropped - their theta/beta still define the pc levels.
MSA_SS_STALE = {(1, 5), (33, 5), (59, 5)}

# OpenQuake switches. RUN_OQ = False on this machine: section 5 then only WRITES the
# job files. Set True on the OQ machine to run-or-reuse + extract.
RUN_OQ = False
DRY_RUN = False         # with RUN_OQ: report what would run, launch nothing
FORCE_RERUN = False     # with RUN_OQ: re-run even when the manifest says current
NEW_WINDOW = False      # with RUN_OQ: own console per calculation (Windows)

# GCIM cache: recompute every GCIM entry used, ignoring the saved ones.
FORCE_RECOMPUTE_GCIM = False

# Refuse to choose pc levels if the local disaggregation lacks IMLs that the
# disaggregation grid lists (i.e. `dvc pull` has not been run).
REQUIRE_CURRENT_DISAGG = True

# -----------------------------------------------------------------------------
# PATHS
# -----------------------------------------------------------------------------
BOOTSTRAP_PTH = Path(cfg["proc_data"]["bootstrapping"])
OUT_PTH = BOOTSTRAP_PTH.parent                   # .../10_fragility_curve_statistical_analysis
INT_PTH = OUT_PTH / "regression_coefficients"    # every intermediate product of this notebook
INT_PTH.mkdir(parents=True, exist_ok=True)

WP1_DIR = Path(cfg["hazard_models"]["eshm20_wp1"])
PSHA_MANIFEST_FP = Path(cfg["hazard_models"]["eshm20_wp1_psha_manifest"])
DISAGG_MANIFEST_FP = Path(cfg["hazard_models"]["eshm20_wp1_disagg_manifest"])

SHARD_DIR = Path(cfg["proc_data"]["AvgSA_03_disagg_data_shards"])
DISAGG_STATS_FP = Path(cfg["proc_data"]["AvgSA_03_disagg_stats_gm_selection"])
DISAGG_GRID_FP = Path(cfg["proc_data"]["disagg_imls_AvgSA_03"])
GCIM_SRC_FP = Path(cfg["proc_data"]["gcim_dists"]) / f"gcim_dist_{IM_TAG}.pickle"

# OpenQuake jobs written by this notebook (section 5) and their extracted products.
SA_BASE_CFG_FP = WP1_DIR / "config_SA_psha_eps4.ini"
SA_NAME = f"SA_0to3_psha_eps{TRUNCATION}"
SA_CFG_FP = WP1_DIR / f"config_{SA_NAME}.ini"
SA_CURVES_FP = INT_PTH / f"SA_hazard_curves_{N_SITES}sites_{TRUNCATION}sig_0to3s.pickle"

D475_NAME = f"{IM_TAG}_disagg_eps{TRUNCATION}_{DESIGN_LEVEL}"
D475_CFG_FP = WP1_DIR / f"config_{D475_NAME}.ini"
D475_FP = INT_PTH / f"{IM_TAG}_disagg_{DESIGN_LEVEL}_eps{TRUNCATION}.pickle"

# GCIM distributions used here (existing + newly computed), re-saved per the brief.
GCIM_PC_FP = INT_PTH / f"gcim_dist_{IM_TAG}_pc_levels.pickle"
GCIM_475_FP = INT_PTH / f"gcim_dist_{IM_TAG}_{DESIGN_LEVEL}.pickle"

FINAL_FP = OUT_PTH / f"site_regression_variables_{IM_TAG}.csv"
COLDICT_FP = INT_PTH / f"site_regression_variables_{IM_TAG}_columns.csv"

## 1. Row skeleton

The 120 (site, structure) rows and their design-group labels come from nb 072's regression dataset. Taking
them from there guarantees that the covariates line up row for row with the responses 073 fits.

In [ ]:
dataset = pd.read_csv(fm.dataset_csv_path(BOOTSTRAP_PTH, IM_TAG))
rows = (dataset[["site", "n_storeys", "structure_id", "tag", "design_group_id"]]
        .sort_values(["n_storeys", "site"])
        .reset_index(drop=True))

assert len(rows) == N_ROWS, f"{len(rows)} rows, expected {N_ROWS}"
assert not rows.duplicated(["site", "n_storeys"]).any(), "duplicate (site, n_storeys)"
assert rows["site"].nunique() == N_SITES
assert rows["design_group_id"].nunique() == N_GROUPS, (
    f"{rows['design_group_id'].nunique()} design groups, expected {N_GROUPS}")
print(f"{len(rows)} rows, {rows['site'].nunique()} sites, "
      f"{rows['design_group_id'].nunique()} design groups")
rows.head()

## 2. Periods: modal `T1` and design `T1_design`

* **`T1`**: the first-mode period of the *nonlinear analysis model*. `ops.eigen` computes it at IDA set-up
  (`templates/template_config_im_SA.py`) and uses it as the IM `SA(T1)` of the IDA SA fragility. The only local
  copy is the `intensity_measure` string of those fragility JSONs, e.g.
  `SpectralAcceleration(period=0.6498…, damping_ratio=0.05)`. It is parsed with the same regex as nb 053.
  It is a property of the design, so it must be constant within a design group.
* **`T1_design`**: the period of the design script's elastic model (`site_designs_summary.csv`, column
  `T`). It usually differs from `T1` because the two models differ, e.g. in how the braces and connections are
  modelled.

In [ ]:
# Same pattern as nb 053: the period is the only float argument named `period=`.
PERIOD_RE = re.compile(r"period=([0-9.eE+-]+)")
FRAG_DIR = Path(cfg["proc_data"]["wp1_sites_fragility_curves"])


def modal_period(site: int, n_storeys: int) -> float:
    # The IDA SA fragility JSON of this row; nb 053 copied the group's file to every
    # member site, so every row has one.
    fp = FRAG_DIR / f"site_{site}" / f"{n_storeys}s_cbf_dc2_site{site}_ida_femap695_collapsefragility_SA.json"
    with open(fp) as f:
        im_str = json.load(f)["intensity_measure"]
    m = PERIOD_RE.search(im_str)
    if m is None:
        raise ValueError(f"no period in {fp.name}: {im_str!r}")
    return float(m.group(1))


rows["T1"] = [modal_period(s, n) for s, n in zip(rows["site"], rows["n_storeys"])]

# Design period, joined on the structure tag ("3s_cbf_dc2_site0", ...).
designs = pd.read_csv(cfg["models"]["site_specific_designs_summary"])
rows = rows.merge(designs[["tag", "T"]].rename(columns={"T": "T1_design"}),
                  on="tag", how="left", validate="1:1")

# T1 belongs to the design, so every site in a group must report the same value.
spread = rows.groupby("design_group_id")["T1"].agg(lambda x: x.max() - x.min())
assert (spread < 1e-9).all(), f"T1 varies within groups:\n{spread[spread >= 1e-9]}"
assert rows[["T1", "T1_design"]].notna().all().all()

print(f"{rows.groupby('design_group_id')['T1'].first().nunique()} distinct T1 over {N_GROUPS} groups")
print(rows.assign(ratio=rows["T1"] / rows["T1_design"])
          .groupby("n_storeys")[["T1", "T1_design", "ratio"]]
          .describe().T.round(3))

## 3. Hazard-curve coefficients `k0, k1, k2`

These come from nb 004 §6: a least-squares fit, in log-log space, of the 4σ mean AvgSA$_{0-3}$ hazard curve over
roughly the 10–10 000 yr band (`rtp_low`/`rtp_high` hold the band actually used). The form is Vamvatsikos (2013)
(`hazrisk.hazard.second_order_approx`):

$$H(x) = k_0 \exp\!\left(-k_1 \ln x - k_2 \ln^2 x\right),\qquad x\ \text{in g},\ H\ \text{in 1/yr}.$$

Note that $k_0$ is a rate, not a log-rate, so it spans orders of magnitude. Use $\ln k_0$ in a regression.

In [ ]:
fits = (pd.read_csv(cfg["proc_data"]["AvgSA_03_so_hc_fits"])
        .rename(columns={"site_id": "site"})[["site", "k0", "k1", "k2", "r2"]])
assert len(fits) == N_SITES and fits["site"].is_unique

rows = rows.merge(fits, on="site", how="left", validate="m:1")
rows["ln_k0"] = np.log(rows["k0"])       # the regression-ready form of k0
fits.describe().T.round(4)

## 4. AvgSA$_{0-3}$ hazard levels and the hazard-curve helpers

Every reading of a hazard curve in this notebook, in either direction, interpolates **linearly in log-log
space**. That is the only reasonable way to read a curve whose ordinate spans many orders of magnitude between
the 25 tabulated points. Two rules apply:

* The zero tail, where the curve sits beyond the 4σ truncation ceiling, is dropped first. A level outside the
  remaining positive range gives **NaN, never an extrapolation**.
* `iml_at_rtp` below does the same thing as `site_hazard_at_return_periods` (used for the table of site levels)
  but works on a single curve. The first check below shows that the two agree.

In [ ]:
def _positive(hc):
    # Rows of an (n, 2) [IML, MAFE] curve with MAFE > 0, i.e. without the zero tail
    # beyond the truncation ceiling (log of zero is undefined).
    hc = np.asarray(hc, dtype=float)
    return hc[hc[:, 1] > 0.0]


def iml_at_rtp(hc, rtp):
    # IML [g] at return period `rtp` [yr]: invert the curve at MAFE = 1/rtp.
    # np.interp needs an ascending abscissa, and MAFE descends along the curve,
    # hence the reversal. NaN when 1/rtp is outside the positive part of the curve.
    pos = _positive(hc)
    if len(pos) < 2 or not np.isfinite(rtp):
        return np.nan
    target = 1.0 / rtp
    if not (pos[:, 1].min() <= target <= pos[:, 1].max()):
        return np.nan
    return float(np.exp(np.interp(np.log(target), np.log(pos[::-1, 1]), np.log(pos[::-1, 0]))))


def rtp_at_iml(hc, iml):
    # Return period [yr] of intensity `iml` [g]: forward reading of the curve.
    # NaN when iml is outside the positive part (in particular in the gap between
    # the last positive tabulated IML and the first zero).
    pos = _positive(hc)
    if len(pos) < 2 or not (pos[0, 0] <= iml <= pos[-1, 0]):
        return np.nan
    return float(1.0 / np.exp(np.interp(np.log(iml), np.log(pos[:, 0]), np.log(pos[:, 1]))))


avgsa_hcs = pd.read_pickle(cfg["proc_data"]["AvgSA_03_hazard_curves_4sig"])
avgsa_hc = {s: avgsa_hcs[s][IMT]["mean"] for s in range(N_SITES)}

# Site hazard levels via the library function.
haz_levels = site_hazard_at_return_periods(avgsa_hcs, range(N_SITES), HAZARD_RTPS, im_key=IMT)
haz_levels.columns = [f"avgsa03_rtp{r}" for r in HAZARD_RTPS]

# Cross-check: the single-curve helper reproduces the library function exactly,
# and a forward-then-inverse round trip returns the return period.
for r in HAZARD_RTPS:
    mine = np.array([iml_at_rtp(avgsa_hc[s], r) for s in range(N_SITES)])
    assert np.allclose(mine, haz_levels[f"avgsa03_rtp{r}"], equal_nan=True)
    back = np.array([rtp_at_iml(avgsa_hc[s], m) for s, m in enumerate(mine)])
    assert np.allclose(back[np.isfinite(back)], r, rtol=1e-9)

print("NaN (return period beyond the positive part of the curve):")
print(haz_levels.isna().sum().to_string())
rows = rows.merge(haz_levels.rename_axis("site").reset_index(), on="site", how="left", validate="m:1")
haz_levels.describe().T.round(4)

## 5. OpenQuake jobs (written here, run on the OQ machine)

**(a) SA PSHA to 3 s**: `config_SA_0to3_psha_eps4.ini`. This is `config_SA_psha_eps4.ini` word for word,
except for the IMT list (PGA plus SA every 0.05 s up to 3.00 s, the same `logscale(0.0005, 5.00, 25)` levels)
and a unique description. It is generated *from* the old file, so any later edit to the old file carries across
and changes the fingerprint. The ESHM20 GMMs in `gmpe_logic_tree_SA_median_branch.xml` (Kotha et al. 2020 up to
8 s, BC Hydro ESHM20 up to 10 s) cover the range.

**(b) Disaggregation at 475 yr**: `config_AvgSA_03_disagg_eps4_rtp475.ini`. It uses the same settings as the
IML-based runs of nb 020 (`oq_runner` template: TRT_Mag_Dist_Eps, 0.2 magnitude bins, 10 km distance bins,
6 ε bins, `num_rlzs_disagg = 0` so that the *mean* disaggregation is produced). The one difference: it uses
`poes_disagg` with the AvgSA IMTL grid instead of `iml_disagg`. OpenQuake then finds **each site's own 475-yr
level** on its mean curve (`hmap3`) and disaggregates there. That makes it one job instead of 60.

**Manifest check.** `oq_runner.run_or_reuse` fingerprints the config, the GMPE and SSC logic trees, the site
model, and the bytes of the source-model tree. It reuses the recorded `calc_id` when all of them match and the
datastore still exists, and otherwise runs the job. The two jobs are recorded in the existing PSHA and
disaggregation manifests.

**Extraction** runs in the same cell, so it happens where the datastores live. The products are small pickles
with a `.manifest.json` sidecar holding the job's input fingerprint and `calc_id`. Extraction is skipped when the
pickle already carries the current fingerprint and `calc_id`.

In [ ]:
def write_text_if_changed(fp: Path, text: str) -> Path:
    # Rewrite only when the content differs, so the file's mtime stays stable
    # (the fingerprint hashes bytes anyway; this just avoids pointless churn).
    fp = Path(fp)
    if not (fp.is_file() and fp.read_text() == text):
        fp.write_text(text)
    return fp


# ---- (a) SA PSHA to 3 s ------------------------------------------------------
# Generated from the existing hand-written SA config: swap the IMT dict and the
# description, keep every other line verbatim.
base = SA_BASE_CFG_FP.read_text()
imts = ["PGA"] + [f"SA({T:.2f})" for T in SA_PSHA_PERIODS]
indent = " " * len("intensity_measure_types_and_levels = {")
imtl_block = ("intensity_measure_types_and_levels = {"
              + f",\n{indent}".join(f'"{imt}": logscale(0.0005, 5.00, 25)' for imt in imts)
              + "}\n")
# The IMT dict holds no nested braces (only logscale(...)), so the first '}' closes it.
sa_text, n_sub = re.subn(r"intensity_measure_types_and_levels\s*=\s*\{.*?\}\n", imtl_block,
                         base, count=1, flags=re.S)
assert n_sub == 1, "IMT block not found in the base SA config"
# Unique, front-loaded description: oq_runner recovers the calc_id from the engine
# listing by (truncated) description when it cannot scrape the log.
sa_text, n_sub = re.subn(r"^description\s*=.*$", f"description = [SA 0-3s psha eps{TRUNCATION} - ESHM20 all sites]",
                         sa_text, count=1, flags=re.M)
assert n_sub == 1, "description line not found in the base SA config"
sa_text = ("# GENERATED by notebooks/03_WP1_ground_motion_set/078-site_regression_variables.ipynb\n"
           f"# from {SA_BASE_CFG_FP.name}: identical except the IMTs (PGA + SA 0.05-3.00 s) and the description.\n"
           + sa_text)
write_text_if_changed(SA_CFG_FP, sa_text)

# ---- (b) AvgSA_03 disaggregation at each site's 475-yr level -----------------
# Mirrors oq_runner._DISAGG_CONFIG_TEMPLATE; the differences are forced by the
# engine: poes_disagg (not iml_disagg) needs the IMTLs to find each site's level,
# and the two are mutually exclusive (oqvalidation.py). `poes` is left unset:
# the engine copies it from poes_disagg. investigation_time = 1, so the PoE is
# 1 - exp(-1/475) and the level is at MAFE = 1/475 exactly.
D475_TEMPLATE = Template('''\
# AvgSA[0,3] disaggregation at each site's own ${rtp}-yr level (poes_disagg).
# Same source model, GMM logic tree and disaggregation bins as the iml_disagg runs
# of nb 020 (oq_runner._DISAGG_CONFIG_TEMPLATE).
#
# GENERATED by notebooks/03_WP1_ground_motion_set/078-site_regression_variables.ipynb

[general]
description = $description
calculation_mode = disaggregation
random_seed = 23

[geometry]

[logic_tree]

[erf]
rupture_mesh_spacing = 5
complex_fault_mesh_spacing = 50
width_of_mfd_bin = 0.1
area_source_discretization = 10.0
pointsource_distance = 75
ps_grid_spacing = 50

[site_params]
reference_vs30_type = measured
reference_vs30_value = 800.0
site_model_file = $site_model_file

reference_depth_to_1pt0km_per_sec = 30.0

[calculation]
source_model_logic_tree_file = $source_model_logic_tree_file
gsim_logic_tree_file = $gsim_logic_tree_file

investigation_time = 1.0

intensity_measure_types_and_levels = {"AvgSA": logscale(0.0005, 3.00, 25)}

truncation_level = $truncation_level
maximum_distance = {'Craton': 400.0,
                    'Subduction Interface': 250.0,
                    'Subduction Inslab': 300.0,
                    'Non-Subduction Deep':500.0,
                    'Volcanic':50.0,
                    'default': 200.0}

[output]
mean_hazard_curves = true

[disaggregation]
poes_disagg = $poe
disagg_outputs = TRT_Mag_Dist_Eps
max_sites_disagg = 60
num_rlzs_disagg = 0
mag_bin_width = 0.2
distance_bin_width = 10.0
coordinate_bin_width = 5
num_epsilon_bins = 6
''')
d475_poe = 1.0 - np.exp(-1.0 / DESIGN_RTP)       # annual PoE of the 475-yr level
write_text_if_changed(D475_CFG_FP, D475_TEMPLATE.substitute(
    rtp=DESIGN_RTP,
    description=f"[disagg AvgSA03 eps{TRUNCATION} rtp{DESIGN_RTP} - ESHM20 all sites]",
    site_model_file=oq_runner.SITE_MODEL_FILE,
    source_model_logic_tree_file=oq_runner.SOURCE_MODEL_LOGIC_TREE_FILE,
    gsim_logic_tree_file=oq_runner.GMPE_LOGIC_TREES["03"],
    truncation_level=TRUNCATION,
    poe=f"{d475_poe:.6g}",
))

for fp in (SA_CFG_FP, D475_CFG_FP):
    print(f"{fp.name}: {len(fp.read_text().splitlines())} lines  (fingerprintable: "
          f"{bool(oq_runner.fingerprint_calc(fp, WP1_DIR))})")

In [ ]:
# ---- extraction functions (used only on the OQ machine) ---------------------
def extract_sa_curves(dstore) -> dict:
    # curves[site][imt][stat] -> (25, 2) [IML g, MAFE 1/yr]: the format of the
    # existing SA_hazard_curves_60sites_4sig.pickle (nb 005).
    curves = oqhelpers.get_hcurves_from_dstore(dstore, mafe=True)
    assert len(curves) == N_SITES, f"{len(curves)} sites in the datastore"
    return curves


def extract_disagg_at_poe(dstore, disagg_type="TRT_Mag_Dist_Eps") -> dict:
    # Per-site disaggregation at the single poes_disagg level, reshaped to the
    # schema of the nb 021 shards so the GCIM driver consumes it unchanged:
    # TRT, Mag, Dist, Eps, Z, P(X>x|T,m), nu_m, P(m|X>x), P(m|X=x).
    #
    # Built from the same oqhelpers building blocks as the IML-based reader. The
    # site's IML is the engine's own `hmap3` (N, M, P): the level at which it
    # actually disaggregated. The occurrence slope P(m|X=x) comes from the job's
    # own mean hazard curve, which covers the full AvgSA grid (unlike an iml_disagg
    # run, whose curve is a single point).
    t = dstore["oqparam"].investigation_time
    data = dstore["disagg-stats"][disagg_type]
    shape_descr = data.attrs["shape_descr"]
    bins = oqhelpers.get_bins(dstore, disagg_type)
    assert len(bins["poe"]) == 1 and bins["imt"] == [IMT], (bins["poe"], bins["imt"])
    hmap3 = np.asarray(dstore["hmap3"])

    out = {"disagg": {}, "iml": {}, "summary": []}
    for site in range(data.shape[0]):
        imtl = float(hmap3[site, 0, 0])
        df = oqhelpers._disagg_array_to_df(
            data[oqhelpers._build_slice(site, 0, 0, shape_descr)], disagg_type, bins)
        hc = oqhelpers.get_hcurve_from_dstore(dstore, site, IMT, "mean", t, mafe=True)
        df = oqhelpers._get_occurence_disagg_from_hc(df, hc, imtl, t)
        out["disagg"][site] = df
        out["iml"][site] = imtl
        out["summary"].append({"site_id": site, "imtl": imtl, "poe": float(bins["poe"][0])}
                              | oqhelpers._disagg_summary_stats(df))
    out["summary"] = pd.DataFrame(out["summary"])
    assert len(out["disagg"]) == N_SITES
    return out


OQ_JOBS = [
    # (manifest key, job file, manifest, product pickle, extractor)
    (SA_NAME, SA_CFG_FP, PSHA_MANIFEST_FP, SA_CURVES_FP, extract_sa_curves),
    (D475_NAME, D475_CFG_FP, DISAGG_MANIFEST_FP, D475_FP, extract_disagg_at_poe),
]

if RUN_OQ:
    from openquake.commonlib.datastore import read

    for name, cfg_fp, manifest_fp, out_fp, extract in OQ_JOBS:
        # Launch only if missing / inputs changed / datastore gone.
        calc_id = oq_runner.run_or_reuse(name, cfg_fp, WP1_DIR, manifest_fp,
                                         force_rerun=FORCE_RERUN, dry_run=DRY_RUN,
                                         new_window=NEW_WINDOW)
        if calc_id is None:          # dry run
            continue
        # Product provenance = the job's input fingerprint + the calc it came from.
        fp_dict = {**oq_runner.fingerprint_calc(cfg_fp, WP1_DIR),
                   **fingerprint(calc_id=int(calc_id))}
        if out_fp.is_file() and manifest_matches(out_fp, fp_dict):
            print(f"[extract] {out_fp.name} current (calc_id={calc_id})")
            continue
        dstore = read(int(calc_id))
        try:
            product = extract(dstore)
        finally:
            dstore.close()
        with open(out_fp, "wb") as f:
            pickle.dump(product, f)
        write_manifest(out_fp, fp_dict)
        print(f"[extract] wrote {out_fp.name} (calc_id={calc_id})")
else:
    print("RUN_OQ = False: job files written, nothing launched. Run this cell with "
          "RUN_OQ = True on the OQ machine and copy back:")
    for *_, out_fp, _ in OQ_JOBS:
        print(f"  {out_fp.name} (+ .manifest.json)")

In [ ]:
# ---- load the OQ products (any machine) -------------------------------------
def load_oq_product(out_fp: Path, cfg_fp: Path):
    # The product counts only if its sidecar fingerprint matches the job file
    # and inputs as they are NOW (subset match: calc_id is not part of the
    # current fingerprint). Otherwise None, and the dependent columns stay NaN.
    if not out_fp.is_file():
        print(f"[missing] {out_fp.name}: run section 5 with RUN_OQ = True on the OQ machine "
              f"and copy it (with its .manifest.json) to {out_fp.parent}")
        return None
    if not manifest_matches(out_fp, oq_runner.fingerprint_calc(cfg_fp, WP1_DIR)):
        print(f"[stale] {out_fp.name}: its job inputs have changed since extraction - re-run section 5 "
              f"on the OQ machine")
        return None
    with open(out_fp, "rb") as f:
        product = pickle.load(f)
    print(f"[ok] {out_fp.name}")
    return product


sa_curves_3s = load_oq_product(SA_CURVES_FP, SA_CFG_FP)
disagg_475 = load_oq_product(D475_FP, D475_CFG_FP)

## 6. The four intensity levels per row

**Pc levels.** Each row's MSA site-specific fragility gives the target
$\mathrm{IM}^*_p = \theta\,e^{\beta\,\Phi^{-1}(p)}$. For $p = 0.2, 0.5, 0.8$, $\Phi^{-1}(p) = -0.8416, 0, 0.8416$.
The candidates are **every disaggregated IML at that site**, i.e. the rows of the disaggregation-stats table.
Those already exclude IMLs with zero hazard at the site (beyond the truncation ceiling). The candidate closest to
$\mathrm{IM}^*_p$ in $|\ln \mathrm{IML} - \ln \mathrm{IM}^*_p|$ is chosen. Distance is measured in log space
because the fragility is lognormal.

The estimates are the plain point estimates `theta`, `beta`, not the bias-corrected `_bc` values. The
`msa_ss_stale` rows are kept but flagged.

**Return period of each level.** This is read log-log from the mean AvgSA curve (section 4). The `rtp` column of
the disaggregation-stats table interpolates **lin-lin** and so understates it. The comparison is printed below
for reference only. An IML that lies between the last positive tabulated point and the truncation zeros has no
log-log return period and is left NaN.

**Design level.** `iml_rtp475` = `avgsa03_rtp475` from section 4.

In [ ]:
# ---- guard: the local disaggregation must be the complete, current one ------
grid = oq_runner.load_imls(DISAGG_GRID_FP)             # the 27 IMLs disaggregated
index = read_index(SHARD_DIR)                         # {site: {imt: [native iml keys]}}
missing_grid = [x for x in grid
                if not all(any(np.isclose(x, k) for k in index[s][IMT]) for s in range(N_SITES))]
if missing_grid:
    msg = (f"disaggregation shards lack IML(s) {missing_grid} that {DISAGG_GRID_FP.name} lists - "
           f"the local data predates the DVC-tracked version. Run `dvc pull` first.")
    if REQUIRE_CURRENT_DISAGG:
        raise RuntimeError(msg)
    print("WARNING (provisional run):", msg)

disagg_stats = pd.read_pickle(DISAGG_STATS_FP)
cands = disagg_stats.loc[disagg_stats["imt"] == IMT, ["site_id", "imtl", "rtp"]]

# ---- MSA site-specific fragility, per row -----------------------------------
est = pd.read_csv(mra.estimates_csv_path(BOOTSTRAP_PTH, "site_msa", IM_TAG, "by_site"))
rows = rows.merge(est[["site", "n_storeys", "theta", "beta"]]
                  .rename(columns={"theta": "msa_ss_theta", "beta": "msa_ss_beta"}),
                  on=["site", "n_storeys"], how="left", validate="1:1")
assert rows[["msa_ss_theta", "msa_ss_beta"]].notna().all().all()
rows["msa_ss_stale"] = [(s, n) in MSA_SS_STALE for s, n in zip(rows["site"], rows["n_storeys"])]

# ---- choose the level IMLs --------------------------------------------------
for lv, p in PC_LEVELS.items():
    target = rows["msa_ss_theta"] * np.exp(rows["msa_ss_beta"] * norm.ppf(p))
    chosen, rtp_ll, rtp_lin = [], [], []
    for site, tgt in zip(rows["site"], target):
        c = cands[cands["site_id"] == site]
        i = np.argmin(np.abs(np.log(c["imtl"].to_numpy()) - np.log(tgt)))
        iml = float(c["imtl"].iloc[i])
        chosen.append(iml)
        rtp_ll.append(rtp_at_iml(avgsa_hc[site], iml))
        rtp_lin.append(float(c["rtp"].iloc[i]))
    rows[f"iml_target_{lv}"] = target
    rows[f"iml_{lv}"] = chosen
    rows[f"rtp_{lv}"] = rtp_ll
    rows[f"_rtp_linlin_{lv}"] = rtp_lin        # comparison only, dropped before saving

rows[f"iml_{DESIGN_LEVEL}"] = rows[f"avgsa03_rtp{DESIGN_RTP}"]
rows[f"rtp_{DESIGN_LEVEL}"] = float(DESIGN_RTP)

# ---- report ------------------------------------------------------------------
for lv in PC_LEVELS:
    miss = np.abs(np.log(rows[f"iml_{lv}"] / rows[f"iml_target_{lv}"]))
    ratio = rows[f"rtp_{lv}"] / rows[f"_rtp_linlin_{lv}"]
    print(f"{lv}: |ln(chosen/target)| median {miss.median():.3f}, max {miss.max():.3f}; "
          f"{rows[f'iml_{lv}'].nunique()} distinct IMLs; "
          f"rtp NaN (beyond positive curve) {rows[f'rtp_{lv}'].isna().sum()}; "
          f"log-log / lin-lin rtp: median {ratio.median():.2f}, max {ratio.max():.2f}")

## 7. Spectral shape from the hazard (UHS-based)

For each row and level, the uniform-hazard ordinate $\mathrm{SA}(T)$ at the level's return period is found by
inverting each period's SA hazard curve at MAFE = 1/rtp. Between the tabulated periods the spectrum is then
interpolated log-log in period, which assumes a straight line between neighbouring points on a log-log spectrum
plot.

* `sa_t1_L` = $\mathrm{SA}(T_1)$. It uses the 0–3 s curves when available and otherwise the existing 0–1.2 s
  curves, which is enough because $T_1 \le 0.91$ s.
* `sa_t1_avgsa03_L` = `sa_t1_L / iml_L`.
* `saratio_uhs_L` needs the 0–3 s curves. It is NaN until `SA_hazard_curves_60sites_4sig_0to3s.pickle` exists.

A return period beyond an SA curve's positive range makes the ordinate, and hence the ratio, NaN. This happens
mainly for the `pc80` level at low-hazard sites, whose collapse intensities have return periods well beyond
10⁴ yr.

In [ ]:
SA_KEY_RE = re.compile(r"^SA\(([0-9.]+)\)$")
sa_curves_12s = pd.read_pickle(cfg["proc_data"]["SA_hazard_curves_4sig"])   # 0 - 1.2 s (nb 005)


def sa_period_keys(curves_site: dict):
    # (periods, keys) of the SA curves of one site, ascending; PGA left out
    # because period 0 has no logarithm (the SaRatio band starts at 0.2 T1 > 0.05 s).
    pairs = sorted((float(m.group(1)), k) for k in curves_site if (m := SA_KEY_RE.match(k)))
    return np.array([p for p, _ in pairs]), [k for _, k in pairs]


def uhs_at(curves_site: dict, rtp: float, T_query) -> np.ndarray:
    # UHS ordinates [g] at periods `T_query` for return period `rtp`: invert each
    # tabulated period's mean curve, then interpolate log-log in period. A query
    # outside the tabulated period range, or a period whose curve does not reach
    # this rtp, returns NaN (no extrapolation in either direction).
    T_grid, keys = sa_period_keys(curves_site)
    sa = np.array([iml_at_rtp(curves_site[k]["mean"], rtp) for k in keys])
    T_query = np.atleast_1d(np.asarray(T_query, dtype=float))
    out = np.full(T_query.shape, np.nan)
    ok = np.isfinite(sa)
    if ok.sum() < 2:
        return out
    Tg, lsa = T_grid[ok], np.log(sa[ok])
    inside = (T_query >= Tg.min()) & (T_query <= Tg.max())
    # A NaN ordinate strictly inside the grid would be bridged by np.interp, so
    # require the whole tabulated span to be finite before interpolating.
    if not ok[(T_grid >= T_query.min()) & (T_grid <= T_query.max())].all():
        inside[:] = False
    out[inside] = np.exp(np.interp(np.log(T_query[inside]), np.log(Tg), lsa))
    return out


def saratio_periods(T1: float) -> np.ndarray:
    # Zhong et al. (2022) Eq. 2: Ti from 0.2 T1 to 3 T1 every 0.01 s (end included).
    return np.arange(SARATIO_TA * T1, SARATIO_TB * T1 + SARATIO_DT / 2, SARATIO_DT)


# The 0-3 s curves supersede the 0-1.2 s ones wherever they exist.
curves_T1 = sa_curves_3s if sa_curves_3s is not None else sa_curves_12s
if sa_curves_3s is not None:
    # They come from the same model, so on the shared periods they must agree.
    T_old, k_old = sa_period_keys(sa_curves_12s[0])
    for s in range(N_SITES):
        T_new, k_new = sa_period_keys(sa_curves_3s[s])
        for T, k in zip(T_old, k_old):
            j = int(np.argmin(np.abs(T_new - T)))
            assert np.isclose(T_new[j], T), f"period {T} missing from the 0-3 s run"
            assert np.allclose(sa_curves_3s[s][k_new[j]]["mean"], sa_curves_12s[s][k]["mean"],
                               rtol=1e-6, atol=0), f"site {s} {k}: 0-3 s run differs from the old curves"
    print("0-3 s SA curves agree with the old 0-1.2 s curves on every shared period")
    assert max(sa_period_keys(sa_curves_3s[0])[0]) >= SARATIO_TB * rows["T1"].max()

for lv in LEVELS:
    sa_t1, sr = [], []
    for site, T1, rtp in zip(rows["site"], rows["T1"], rows[f"rtp_{lv}"]):
        if not np.isfinite(rtp):
            sa_t1.append(np.nan); sr.append(np.nan); continue
        sa_t1.append(uhs_at(curves_T1[site], rtp, T1)[0])
        if sa_curves_3s is None:
            sr.append(np.nan); continue
        num = uhs_at(sa_curves_3s[site], rtp, T1)[0]
        den = uhs_at(sa_curves_3s[site], rtp, saratio_periods(T1))
        # geometric mean of the band; any NaN ordinate makes the ratio NaN
        sr.append(num / np.exp(np.mean(np.log(den))) if np.isfinite(den).all() else np.nan)
    rows[f"sa_t1_{lv}"] = sa_t1
    rows[f"sa_t1_avgsa03_{lv}"] = rows[f"sa_t1_{lv}"] / rows[f"iml_{lv}"]
    rows[f"saratio_uhs_{lv}"] = sr

rows[[f"sa_t1_avgsa03_{lv}" for lv in LEVELS] + [f"saratio_uhs_{lv}" for lv in LEVELS]].describe().T.round(3)

## 8. GCIM-based quantities: duration and conditional-mean spectral shape

Each GCIM distribution (Bradley 2010) is conditioned on AvgSA$_{0-3}$ = level at the site. The `stats` table of
each distribution holds the **ln-space** mixture moments (`mean`, `sigma`) of `RSD595`, `PGA` and 20 `SA(T)`
ordinates (0.025–3 s). They are computed exactly as in nb 031 (`calculate_gcim_distributions_for_sites`, the
selection context of `setup_AvgSA03_gcim_gm_selection`). Recomputing one existing entry reproduces the stored
stats to machine precision, so the in-notebook pipeline and nb 031 give the same results.

* **Pc levels.** The existing `gcim_dist_AvgSA_03.pickle` covers only each site's *analysed* stripes. Any other
  disaggregated level needed here is computed from its disaggregation shard. All entries used, existing and new,
  are re-saved to `regression_coefficients/` with a provenance sidecar (`cache_utils.load_or_compute`). The source
  pickle is not modified.
* **475 yr.** These come from the section-5 disaggregation, once the pickle exists.

**Columns per level:**
* `ln_rsd595_mean_L` and `ln_rsd595_sigma_L`.
* `ln_saratio_cm_L` $= \mu_{\ln SA(T_1)} - \frac1n\sum_i \mu_{\ln SA(T_i)}$, with $\mu$ interpolated linearly
  in $\ln T$ between the GCIM periods.
* `ln_sa_t1_avgsa03_cm_L` $= \mu_{\ln SA(T_1)} - \ln \mathrm{IML}_L$.

In [ ]:
from phd_project.scripts.WP1_ground_motion_set.gm_selection import calculate_gcim_distributions_for_sites
from phd_project.scripts.WP1_ground_motion_set.setup_AvgSA03_gm_selection import (
    setup_AvgSA03_gcim_gm_selection, SELECTION_CONFIG)

_GCIM_CTX = None


def gcim_ctx():
    # Selection context of nb 031 (GMM map, correlation models, site model, ...),
    # built once and only when something actually has to be computed. sites=()
    # loads no disaggregation shard; ~10 s, mostly reading the record database.
    global _GCIM_CTX
    if _GCIM_CTX is None:
        _, stats, site_model, ctx, _ = setup_AvgSA03_gcim_gm_selection(sites=())
        _GCIM_CTX = (stats, site_model, ctx)
    return _GCIM_CTX


def compute_gcim(batch: dict) -> dict:
    # {(site, iml): disagg DataFrame} -> {(site, iml): {"stats", "pdfs", "cdfs"}}
    stats, site_model, ctx = gcim_ctx()
    return calculate_gcim_distributions_for_sites(
        batch, stats, site_model, ctx, SELECTION_CONFIG["percentiles"])


def _find_key(d: dict, site: int, iml: float):
    # Float keys: match on site and np.isclose(iml) rather than exact equality.
    return next((k for k in d if k[0] == site and np.isclose(k[1], iml)), None)


# ---- the (site, iml) pairs the pc levels need --------------------------------
pc_keys = sorted({(int(s), float(x)) for lv in PC_LEVELS
                  for s, x in zip(rows["site"], rows[f"iml_{lv}"])})


def build_pc_gcim() -> dict:
    # Take existing entries from the nb 031 pickle; compute the rest from the
    # shards, loading one site's shard at a time (~70 MB each).
    with open(GCIM_SRC_FP, "rb") as f:
        src = pickle.load(f)
    out, todo = {}, []
    for key in pc_keys:
        hit = None if FORCE_RECOMPUTE_GCIM else _find_key(src, *key)
        if hit is not None:
            out[key] = src[hit]
        else:
            todo.append(key)
    del src
    print(f"{len(pc_keys)} (site, iml) needed: {len(out)} reused from {GCIM_SRC_FP.name}, "
          f"{len(todo)} to compute")
    for site in sorted({s for s, _ in todo}):
        shard = load_shards(SHARD_DIR, {site})[site][IMT]
        batch = {}
        for s, iml in todo:
            if s != site:
                continue
            native = next(k for k in shard if np.isclose(k, iml))
            batch[(s, iml)] = shard[native]
        out.update(compute_gcim(batch))
    return out


# Provenance: the key set, every input file, and the selection configuration.
pc_fp = fingerprint(
    keys=pc_keys,
    gcim_src=GCIM_SRC_FP,
    shard_content_hashes=SHARD_DIR / "_content_hashes.json",
    disagg_stats=DISAGG_STATS_FP,
    site_model=cfg["hazard_models"]["eshm20_wp1_site_model"],
    gmm_lt=cfg["hazard_models"]["eshm20_AvgSA_03_median_lt"],
    selection_config=SELECTION_CONFIG,
    force=FORCE_RECOMPUTE_GCIM,
)
gcim_pc = load_or_compute(GCIM_PC_FP, pc_fp, build_pc_gcim, force_recompute=FORCE_RECOMPUTE_GCIM)
assert all(k in gcim_pc for k in pc_keys)

# ---- 475 yr: from the section-5 disaggregation ------------------------------
gcim_475 = None
if disagg_475 is not None:
    def build_475_gcim() -> dict:
        batch = {(s, disagg_475["iml"][s]): disagg_475["disagg"][s] for s in range(N_SITES)}
        return compute_gcim(batch)

    gcim_475 = load_or_compute(GCIM_475_FP, fingerprint(
        disagg=D475_FP,
        site_model=cfg["hazard_models"]["eshm20_wp1_site_model"],
        gmm_lt=cfg["hazard_models"]["eshm20_AvgSA_03_median_lt"],
        selection_config=SELECTION_CONFIG,
    ), build_475_gcim)

In [ ]:
def gcim_features(stats: pd.DataFrame, T1: float, iml: float) -> dict:
    # Level-specific covariates from one GCIM `stats` table (ln-space moments).
    mu = stats["mean"]
    sa_rows = [(float(m.group(1)), k) for k in mu.index if (m := SA_KEY_RE.match(k))]
    sa_rows.sort()
    lnT = np.log([T for T, _ in sa_rows])
    mu_sa = mu[[k for _, k in sa_rows]].to_numpy()
    Ti = saratio_periods(T1)
    # The GCIM grid (0.025-3 s) must span the SaRatio band (<= 3 x 0.91 s).
    assert lnT[0] <= np.log(Ti[0]) and np.log(Ti[-1]) <= lnT[-1] + 1e-12, (T1, Ti[[0, -1]])
    mu_T1 = np.interp(np.log(T1), lnT, mu_sa)
    mu_Ti = np.interp(np.log(Ti), lnT, mu_sa)
    return {
        "ln_rsd595_mean": float(stats.loc["RSD595", "mean"]),
        "ln_rsd595_sigma": float(stats.loc["RSD595", "sigma"]),
        "ln_saratio_cm": float(mu_T1 - mu_Ti.mean()),     # E[ln SaRatio | IM]
        "ln_sa_t1_avgsa03_cm": float(mu_T1 - np.log(iml)),
    }


FEATURES = ["ln_rsd595_mean", "ln_rsd595_sigma", "ln_saratio_cm", "ln_sa_t1_avgsa03_cm"]

# Pc levels: keyed on the chosen disaggregated IML.
for lv in PC_LEVELS:
    feats = [gcim_features(gcim_pc[(int(s), float(x))]["stats"], T1, x)
             for s, x, T1 in zip(rows["site"], rows[f"iml_{lv}"], rows["T1"])]
    for f in FEATURES:
        rows[f"{f}_{lv}"] = [d[f] for d in feats]

# 475 yr: keyed on the level OpenQuake disaggregated at (hmap3), which should
# equal the log-log reading of section 4. Both are kept, and the GCIM ratio uses
# the IML the GCIM was actually conditioned on.
rows[f"iml_{DESIGN_LEVEL}_oq"] = np.nan
for f in FEATURES:
    rows[f"{f}_{DESIGN_LEVEL}"] = np.nan
if gcim_475 is not None:
    rows[f"iml_{DESIGN_LEVEL}_oq"] = [disagg_475["iml"][s] for s in rows["site"]]
    rel = rows[f"iml_{DESIGN_LEVEL}_oq"] / rows[f"iml_{DESIGN_LEVEL}"] - 1
    print(f"OQ 475-yr level vs log-log reading: max |rel. diff| = {rel.abs().max():.2e}")
    for i, (s, T1) in enumerate(zip(rows["site"], rows["T1"])):
        x = disagg_475["iml"][s]
        d = gcim_features(gcim_475[_find_key(gcim_475, s, x)]["stats"], T1, x)
        for f in FEATURES:
            rows.loc[i, f"{f}_{DESIGN_LEVEL}"] = d[f]

rows[[f"{f}_{lv}" for f in FEATURES for lv in LEVELS]].describe().T.round(3)

## 9. Assemble, validate, save

In [ ]:
# ---- column order: identifiers, structure, site hazard, then per level -------
ID_COLS = ["site", "n_storeys", "structure_id", "tag", "design_group_id"]
STRUCT_COLS = ["T1", "T1_design"]
SITE_COLS = ["k0", "ln_k0", "k1", "k2", "r2", *[f"avgsa03_rtp{r}" for r in HAZARD_RTPS]]
FRAG_COLS = ["msa_ss_theta", "msa_ss_beta", "msa_ss_stale"]
LEVEL_COLS = []
for lv in LEVELS:
    LEVEL_COLS += ([f"iml_target_{lv}"] if lv in PC_LEVELS else [])
    LEVEL_COLS += [f"iml_{lv}"] + ([f"iml_{lv}_oq"] if lv == DESIGN_LEVEL else [])
    LEVEL_COLS += [f"rtp_{lv}", f"sa_t1_{lv}", f"sa_t1_avgsa03_{lv}", f"saratio_uhs_{lv}",
                   f"ln_sa_t1_avgsa03_cm_{lv}", f"ln_saratio_cm_{lv}",
                   f"ln_rsd595_mean_{lv}", f"ln_rsd595_sigma_{lv}"]

reg = rows[ID_COLS + STRUCT_COLS + SITE_COLS + FRAG_COLS + LEVEL_COLS].copy()
assert not set(rows.columns) - set(reg.columns) - {c for c in rows if c.startswith("_")}, \
    "a computed column was not placed in the output order"

# ---- validation ---------------------------------------------------------------
assert len(reg) == N_ROWS and not reg.duplicated(["site", "n_storeys"]).any()
# Both periods belong to the design. Compare with a tolerance: the design script
# reruns per site, so identical designs can differ in the last bits of T.
for c in ("T1", "T1_design"):
    spread = reg.groupby("design_group_id")[c].agg(lambda x: x.max() / x.min() - 1)
    assert (spread < 1e-9).all(), f"{c} differs within a design group:\n{spread[spread >= 1e-9]}"

# Quantities that depend on the site only must agree between its two structures.
site_only = SITE_COLS + [f"iml_{DESIGN_LEVEL}", f"iml_{DESIGN_LEVEL}_oq",
                         f"ln_rsd595_mean_{DESIGN_LEVEL}", f"ln_rsd595_sigma_{DESIGN_LEVEL}"]
for c in site_only:
    per_site = reg.groupby("site")[c].agg(lambda x: np.nanmax(x) - np.nanmin(x) if x.notna().any() else 0.0)
    assert (per_site.abs() < 1e-12).all(), f"{c} differs between the two structures at a site"

# Why each column has NaNs: OQ products not yet available, or a level beyond the
# positive part of a hazard curve (truncation ceiling).
nan_reason = {}
for c in reg.columns:
    n = int(reg[c].isna().sum())
    if n == 0:
        continue
    if c.startswith("saratio_uhs") and sa_curves_3s is None:
        why = f"awaiting {SA_CURVES_FP.name} (section 5)"
    elif c.endswith(DESIGN_LEVEL + "_oq") or (c.endswith(DESIGN_LEVEL) and c.startswith("ln_") and disagg_475 is None):
        why = f"awaiting {D475_FP.name} (section 5)"
    else:
        why = "level beyond the positive part of a hazard curve (truncation ceiling)"
    nan_reason[c] = (n, why)
print(pd.DataFrame(nan_reason, index=["n_nan", "reason"]).T.to_string() if nan_reason else "no NaNs")

In [ ]:
# ---- collinearity preview ------------------------------------------------------
# analysis_models.md s.A3 warns that k0/k1/k2 and spectral shape are collinear. VIF_j
# is the j-th diagonal element of the inverse correlation matrix, i.e.
# 1 / (1 - R^2_j) of regressing covariate j on all the others; it is > 10 when a
# covariate is ~90 %+ explained by the rest. Shown at pc50, for the covariates
# available now (rows with any NaN dropped).
cand_cols = ["T1", "ln_k0", "k1", "k2", f"avgsa03_rtp{DESIGN_RTP}",
             "sa_t1_avgsa03_pc50", "ln_saratio_cm_pc50", "ln_rsd595_mean_pc50", "ln_rsd595_sigma_pc50"]
X = reg[cand_cols].dropna()
corr = X.corr()
vif = pd.Series(np.diag(np.linalg.inv(corr.to_numpy())), index=cand_cols, name="VIF")
print(f"{len(X)} complete rows")
display(corr.round(2))
display(vif.round(1).to_frame())

In [ ]:
# ---- diagnostic plots ----------------------------------------------------------
colours = {3: "tab:blue", 5: "tab:orange"}
panels = [("T1_design", "T1"), ("k1", "k2"),
          ("sa_t1_avgsa03_pc50", "ln_saratio_cm_pc50"),
          ("ln_rsd595_mean_pc20", "ln_rsd595_mean_pc80"),
          ("avgsa03_rtp475", "iml_pc50"), ("T1", "ln_saratio_cm_pc50")]
fig, axs = plt.subplots(2, 3, figsize=(11, 6.5))
for ax, (xc, yc) in zip(axs.flat, panels):
    for n, g in reg.groupby("n_storeys"):
        ax.scatter(g[xc], g[yc], s=14, alpha=0.8, color=colours[n], label=f"{n}-storey")
    ax.set_xlabel(xc); ax.set_ylabel(yc)
    if xc == "T1_design":
        lim = [reg[[xc, yc]].min().min(), reg[[xc, yc]].max().max()]
        ax.plot(lim, lim, "k--", lw=0.8)      # 1:1 line
axs.flat[0].legend(frameon=False, fontsize=8)
fig.suptitle("Site regression variables - selected pairs (120 rows)")
fig.tight_layout()

In [ ]:
# ---- column dictionary ----------------------------------------------------------
LEVEL_TEXT = {
    "pc20": "disaggregated AvgSA_03 IML closest (log) to the MSA-SS fragility's 20 % collapse intensity",
    "pc50": "... 50 % (median) collapse intensity",
    "pc80": "... 80 % collapse intensity",
    DESIGN_LEVEL: f"AvgSA_03 at the {DESIGN_RTP}-yr return period",
}
base_desc = {
    "site": ("site id 0-59", "-", False, "sites.csv row"),
    "n_storeys": ("storeys of the structure (3 or 5)", "-", False, "nb 072 dataset"),
    "structure_id": ("global design id 0-50", "-", False, "nb 072 dataset"),
    "tag": ("structure tag", "-", False, "nb 072 dataset"),
    "design_group_id": ("design group label", "-", False, "nb 072 dataset"),
    "T1": ("first-mode period of the nonlinear OpenSees model", "s", False,
           "IDA SA fragility JSON 'intensity_measure'"),
    "T1_design": ("fundamental period of the design-script elastic model", "s", False,
                  "site_designs_summary.csv column T"),
    "k0": ("2nd-order hazard fit H = k0 exp(-k1 ln x - k2 ln^2 x), AvgSA_03 mean 4sig", "1/yr", False,
           "AvgSA_03_hazard_fit_second_order_60sites_4sig.csv"),
    "ln_k0": ("ln(k0)", "ln(1/yr)", True, "derived"),
    "k1": ("2nd-order hazard fit slope coefficient", "-", False, "as k0"),
    "k2": ("2nd-order hazard fit curvature coefficient", "-", False, "as k0"),
    "r2": ("R^2 of the 2nd-order fit", "-", False, "as k0"),
    "msa_ss_theta": ("MSA site-specific fragility median (defines the pc levels)", "g", False,
                     "site_msa_estimates_AvgSA_03_by_site.csv"),
    "msa_ss_beta": ("MSA site-specific fragility dispersion", "-", False, "as msa_ss_theta"),
    "msa_ss_stale": ("MSA-SS fragility not refitted after an extra stripe (sites 1/33/59 5s)", "-", False,
                     "memory note 2026-09-15"),
}
for r in HAZARD_RTPS:
    base_desc[f"avgsa03_rtp{r}"] = (f"AvgSA_03 mean-hazard IML at {r} yr (log-log)", "g", False,
                                    "AvgSA_03_hazard_curves_60sites_4sig.pickle")
level_desc = {
    "iml_target": ("fragility quantile theta exp(beta Phi^-1(p))", "g", False, "MSA-SS fragility"),
    "iml": ("AvgSA_03 intensity level", "g", False, "disagg grid / hazard curve"),
    "iml_oq": ("level at which OpenQuake disaggregated (hmap3)", "g", False, D475_FP.name),
    "rtp": ("return period of the level, log-log on the AvgSA_03 mean curve", "yr", False, "hazard curve"),
    "sa_t1": ("UHS SA(T1) at the level's return period", "g", False, "SA hazard curves"),
    "sa_t1_avgsa03": ("SA(T1)_UHS / AvgSA_03 level", "-", False, "SA hazard curves"),
    "saratio_uhs": ("SaRatio(0.2T1, T1, 3T1), UHS ordinates, 0.01 s (Zhong et al. 2022 Eq. 2)", "-", False,
                    SA_CURVES_FP.name),
    "ln_sa_t1_avgsa03_cm": ("E[ln SA(T1) | AvgSA_03] - ln(level)", "-", True, "GCIM"),
    "ln_saratio_cm": ("E[ln SaRatio | AvgSA_03]: GCIM conditional-mean spectrum", "-", True, "GCIM"),
    "ln_rsd595_mean": ("GCIM mixture mean of ln Ds5-95 | AvgSA_03", "ln s", True, "GCIM"),
    "ln_rsd595_sigma": ("GCIM mixture std of ln Ds5-95 | AvgSA_03", "ln s", True, "GCIM"),
}
records = []
for c in reg.columns:
    if c in base_desc:
        d, u, is_ln, src = base_desc[c]
    else:
        lv = next(lv for lv in LEVELS if c.endswith("_" + lv) or c.endswith("_" + lv + "_oq"))
        stem = c[: c.rindex("_" + lv)] + ("_oq" if c.endswith("_oq") else "")
        d, u, is_ln, src = level_desc[stem]
        d = f"{d} @ {lv}: {LEVEL_TEXT[lv]}"
    records.append({"column": c, "description": d, "units": u, "ln_space": is_ln, "source": src,
                    "n_nan": int(reg[c].isna().sum())})
coldict = pd.DataFrame(records)

reg.to_csv(FINAL_FP, index=False)
coldict.to_csv(COLDICT_FP, index=False)
print(f"wrote {FINAL_FP}  ({reg.shape[0]} rows x {reg.shape[1]} columns)")
print(f"wrote {COLDICT_FP}")
coldict